# Celda 0

In [1]:
from pathlib import Path
import sys
from datetime import datetime

# Resolver raíz del proyecto: .../TT2
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Rutas principales
DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "refinement_muestra_20"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Identificador del experimento
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_refinement_m20_{RUN_TS}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
DATA_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/data
OUTPUT_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/refinement_muestra_20
EXPERIMENT_ID: exp_refinement_m20_20260321_171527


In [2]:
import json
import pandas as pd

from configs.models import MODELS, DEFAULT_GENERATION_CONFIG
from configs.rules import RULESETS
from src.experiment.runner import ExperimentRunner
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Archivo de muestra para refinamiento
SAMPLE_PATH = DATA_DIR / "Muestra_csv.csv"

df_sample = pd.read_csv(SAMPLE_PATH)

print("Shape:", df_sample.shape)
print("Columnas:", list(df_sample.columns))
display(df_sample.head(3))

Shape: (20, 17)
Columnas: ['id', 'idFinal', 'grupo', 'motivo', 'Segmento', 'Propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


,id,idFinal,grupo,motivo,Segmento,Propuesta,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,Vivian,10,1,4,5.0,NaN,NaN,NaN,NaN,NaN,1
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Vivian,14,5,1,6.0,NaN,NaN,NaN,NaN,NaN,1
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,Vivian,5,6,19,1.0,NaN,NaN,NaN,NaN,NaN,1


In [5]:
META_COLS = ["idFinal", "grupo", "motivo", "lex"]

required_cols = ["id", "Segmento", "Propuesta"]
missing = [c for c in required_cols if c not in df_sample.columns]

if missing:
    raise ValueError(f"Faltan columnas requeridas en la muestra: {missing}")

df_refine = df_sample.copy()

df_refine = df_refine.rename(
    columns={
        "id": "sample_id",
        "Segmento": "source_text",
        "Propuesta": "reference_text",
    }
)

keep_cols = ["sample_id", "source_text", "reference_text"] + [c for c in META_COLS if c in df_refine.columns]
df_refine = df_refine[keep_cols].copy()

print("Shape refinamiento:", df_refine.shape)
display(df_refine.head(5))

Shape refinamiento: (20, 7)


,sample_id,source_text,reference_text,idFinal,grupo,motivo,lex
0,2088,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,2976,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,3430,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1
3,3679,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,1
4,3145,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,1


In [6]:
FEW_SHOT_EXAMPLES = [
    {
        "source": """A continuación una tabla donde se demuestra como crecen los ahorros al pasar los años: Edad Inversión Intereses Saldo Inversión Intereses Saldo Laura Ganados Alberto Ganados 22 $800 $80 $880 23 $800 $168 $1.232 Valor al Jubilarse $311.092 Valor al Jubilarse $263.232 Menos las contribuciones iniciales $6.400 Menos las contribuciones iniciales $28.800 Ganancia Neta $304.692 Ganancia Neta $234.""",
        "target": """A continuación, presentamos una tabla donde se demuestra como crecen los ahorros al pasar los años. Las categorías en que se divide la tabla son edad, inversión, intereses ganados y saldo. La tabla contrasta las ganancias en intereses por año de Laura y Alberto con una proyección de edad de jubilación de sesenta y cinco años y según la cantidad de años que mantuvieron el ahorro."""
    },
    {
        "source": """El varias veces mencionado Yager, nos dice acertadamente sobre estos aspectos lo siguiente: La mayoría de ustedes están hambrientos de tener libertad financiera, están hastiados de tener batallas financieras porque son como un carrusel.""",
        "target": """Yager habla acertadamente sobre estos aspectos y dice que la mayoría ambiciona tener libertad financiera y está cansada de tener batallas financieras inútiles."""
    },
    {
        "source": """De acuerdo con esta medición, la proporción de pobres en la población mundial quienes viven con menos de un dólar por día descendió levemente entre 1987 y 1993, pues pasó del 30% al 29%.""",
        "target": """Según esta medición, la proporción de personas pobres en la población mundial descendió un poco entre mil novecientos ochenta y siete y mil novecientos noventa y tres. Es decir, el porcentaje de personas que vivían con menos de un dólar por día pasó del treinta al veintinueve por ciento."""
    },
    {
        "source": """- Rentabilidad
INTERÉS DE 2 PESOS SOBRE UNA INVERSIÓN DE 100 PESOS = RENTABILIDAD DE 2 % ((2 ÷ 100) × 100 = 2%) POR CADA 100 PESOS SE GANAN 2 PESOS
 Endeudamiento
 DEUDA DE 30 PESOS POR SOBRE EL CAPITAL DE 20 PESOS = ENDEUDAMIENTO DE 1,5 VECES EL CAPITAL (30 ÷ 20 = 1,5) POR CADA PESO DE CAPITAL EXISTE 1,5 PESOS DE DEUDA
 Liquidez
80 PESOS DE DINERO DISPONIBLE POR SOBRE DEUDAS DE 40 PESOS = LIQUIDEZ DE DOS VECES EL VALOR DE LA DEUDA (80 ÷ 40 = 2) POR CADA PESO DE DEUDA SE DISPONE DE 2 PESOS PARA PAGO""",
        "target": """EL INTERÉS DE DOS PESOS SOBRE UNA INVERSIÓN DE CIEN PESOS DA UNA RENTABILIDAD DE DOS POR CIENTO. ES DECIR, POR CADA CIEN PESOS GANAMOS DOS PESOS. LA DEUDA DE TREINTA PESOS SOBRE EL CAPITAL DE VEINTE PESOS RESULTA EN UNA DEUDA DE UNO PUNTO CINCO VECES EL CAPITAL. ES DECIR, POR CADA PESO DE CAPITAL HAY UNO PUNTO CINCO PESOS DE DEUDA. EN OCHENTA PESOS DE DINERO DISPONIBLE SOBRE DEUDAS DE CUARENTA PESOS DA UNA LIQUIDEZ DEL DOBLE DE LA DEUDA. POR CADA PESO ADEUDADO HAY DOS PESOS PARA PAGAR."""
    },
]

FEW_SHOT_EXAMPLE_IDS = [78, 1805, 3635, 5262]

print("Número de ejemplos few-shot:", len(FEW_SHOT_EXAMPLES))
print("IDs few-shot excluidos del corpus formal:", FEW_SHOT_EXAMPLE_IDS)

Número de ejemplos few-shot: 4
IDs few-shot excluidos del corpus formal: [78, 1805, 3635, 5262]


In [7]:
ACTIVE_MODELS = [
    "llama3",
    "mistral",
]

ACTIVE_TECHNIQUES = [
    "zero-shot",
    "few-shot",
]

ACTIVE_RULESETS = [
    "R0",
    "R4",
]

PARAM_GRID = [
    {
        "temperature": 0.2,
        "top_p": 0.85,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
    },
    {
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 256,
    },
]

print("Modelos:", ACTIVE_MODELS)
print("Técnicas:", ACTIVE_TECHNIQUES)
print("Rulesets:", ACTIVE_RULESETS)
print("N configuraciones params:", len(PARAM_GRID))

Modelos: ['llama3', 'mistral']
Técnicas: ['zero-shot', 'few-shot']
Rulesets: ['R0', 'R4']
N configuraciones params: 2


In [8]:
for model_name in ACTIVE_MODELS:
    if model_name not in MODELS:
        raise ValueError(f"Modelo no definido en MODELS: {model_name}")

for ruleset_name in ACTIVE_RULESETS:
    if ruleset_name not in RULESETS:
        raise ValueError(f"Ruleset no definido en RULESETS: {ruleset_name}")

for tech in ACTIVE_TECHNIQUES:
    if tech not in ["zero-shot", "few-shot"]:
        raise ValueError(f"Técnica no soportada: {tech}")

print("Configuración validada correctamente.")

Configuración validada correctamente.


In [9]:
runner = ExperimentRunner(
    experiment_id=EXPERIMENT_ID,
    log_dir=str(PROJECT_ROOT / "outputs" / "logs")
)

print("Runner inicializado:", runner.experiment_id)

Runner inicializado: exp_refinement_m20_20260321_171527


In [11]:
test_row = df_refine.iloc[0]

test_prompt_type = ACTIVE_TECHNIQUES[0]

test_record = runner.run_one(
    dataset_name="muestra_20_refinamiento",
    model_key=ACTIVE_MODELS[0],
    prompt_type=test_prompt_type,
    ruleset="R0",
    source_text=test_row["source_text"],
    reference_text=test_row["reference_text"],
    sample_id=str(test_row["sample_id"]),
    fold_id=None,
    split_name="refinement",
    few_shot_examples=FEW_SHOT_EXAMPLES if test_prompt_type == "few-shot" else None,
    few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if test_prompt_type == "few-shot" else None,
    generation_config=PARAM_GRID[0],
)

test_record.to_dict()

{'experiment_id': 'exp_refinement_m20_20260321_171527',
 'run_id': '39d644ee-09b9-4264-bf04-2aafea87caa0',
 'timestamp': '2026-03-21T17:19:35.607810',
 'dataset_name': 'muestra_20_refinamiento',
 'fold_id': None,
 'split_name': 'refinement',
 'model_key': 'llama3',
 'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'backend': 'ollama',
 'prompt_type': 'zero-shot',
 'ruleset': 'R0',
 'few_shot_example_ids': [],
 'generation_config': {'temperature': 0.2,
  'top_p': 0.85,
  'repetition_penalty': 1.1,
  'max_new_tokens': 256},
 'sample_id': '2088',
 'source_text': 'La comunidad humana más antigua ha sido denominada horda primitiva.',
 'reference_text': 'La comunidad humana más antigua se llamó tribu.',
 'generated_text': 'La comunidad humana más antigua se llama horda primitiva.',
 'prompt_text': 'Reescribe en español el siguiente texto con lenguaje más claro y sencillo.\nConserva el significado original y no inventes información.\n\nDevuelve solo la versión final simplificada.\n\nTexto:

In [12]:
all_records = []

total_runs = (
    len(ACTIVE_MODELS)
    * len(ACTIVE_TECHNIQUES)
    * len(ACTIVE_RULESETS)
    * len(PARAM_GRID)
    * len(df_refine)
)

run_counter = 0

for model_key in ACTIVE_MODELS:
    for prompt_type in ACTIVE_TECHNIQUES:
        for ruleset in ACTIVE_RULESETS:
            for generation_config in PARAM_GRID:
                for _, row in df_refine.iterrows():
                    run_counter += 1
                    print(
                        f"[{run_counter}/{total_runs}] "
                        f"model={model_key} | prompt={prompt_type} | "
                        f"ruleset={ruleset} | sample_id={row['sample_id']}"
                    )

                    record = runner.run_one(
                        dataset_name="muestra_20_refinamiento",
                        model_key=model_key,
                        prompt_type=prompt_type,
                        ruleset=ruleset,
                        source_text=row["source_text"],
                        reference_text=row["reference_text"],
                        sample_id=str(row["sample_id"]),
                        fold_id=None,
                        split_name="refinement",
                        few_shot_examples=FEW_SHOT_EXAMPLES if prompt_type == "few-shot" else None,
                        few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if prompt_type == "few-shot" else None,
                        generation_config=generation_config,
                    )

                    record_dict = record.to_dict()

                    # Conservar metadatos de la muestra para análisis posterior
                    for meta_col in ["idFinal", "grupo", "motivo", "lex"]:
                        if meta_col in row.index:
                            record_dict[meta_col] = row[meta_col]

                    all_records.append(record_dict)

print(f"Corridas completadas: {len(all_records)}")

[1/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=2088
[2/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=2976
[3/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3430
[4/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3679
[5/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3145
[6/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=507
[7/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=1756
[8/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3093
[9/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3192
[10/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3525
[11/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3627
[12/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3206
[13/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=329
[14/320] model=llama3 | prompt=zero-shot | ruleset=R0 | sample_id=3481
[15/320] model=ll

In [13]:
results_df = pd.DataFrame(all_records)

print("Shape results_df:", results_df.shape)
display(results_df.head(3))

if "status" in results_df.columns:
    display(results_df["status"].value_counts(dropna=False))

Shape results_df: (320, 26)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,generated_text,prompt_text,inference_seconds,status,error_message,metrics,idFinal,grupo,motivo,lex
0,exp_refinement_m20_20260321_171527,ac09fbe2-f126-47cc-8d1f-c4e58ee7ac4c,2026-03-21T17:22:27.934160,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,La comunidad humana más antigua se llama horda...,Reescribe en español el siguiente texto con le...,9.4107,success,None,{},1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_refinement_m20_20260321_171527,b0cd4bff-0834-4169-85f8-33278c9b71c4,2026-03-21T17:22:37.517136,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,Pon $1 al día y también mete el cambio en un s...,Reescribe en español el siguiente texto con le...,7.4372,success,None,{},881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_refinement_m20_20260321_171527,6fa8b84e-4ddf-474d-a9b5-0fb0b0f90bff,2026-03-21T17:22:45.087748,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,Un problema importante en los negocios es dete...,Reescribe en español el siguiente texto con le...,7.0582,success,None,{},2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


status
success    320
Name: count, dtype: int64

In [14]:
raw_results_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_raw_results.csv"

results_df.to_csv(raw_results_path, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(raw_results_path)

Archivo guardado en:
/home/harielpadillasanchez/Documentos/TT/TT2/outputs/refinement_muestra_20/exp_refinement_m20_20260321_171527_raw_results.csv


In [15]:
successful_df = results_df[results_df["status"] == "success"].copy()

print("Corridas exitosas:", len(successful_df))
print("Corridas con error:", len(results_df) - len(successful_df))

display(successful_df.head(3))

Corridas exitosas: 320
Corridas con error: 0


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,generated_text,prompt_text,inference_seconds,status,error_message,metrics,idFinal,grupo,motivo,lex
0,exp_refinement_m20_20260321_171527,ac09fbe2-f126-47cc-8d1f-c4e58ee7ac4c,2026-03-21T17:22:27.934160,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,La comunidad humana más antigua se llama horda...,Reescribe en español el siguiente texto con le...,9.4107,success,None,{},1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_refinement_m20_20260321_171527,b0cd4bff-0834-4169-85f8-33278c9b71c4,2026-03-21T17:22:37.517136,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,Pon $1 al día y también mete el cambio en un s...,Reescribe en español el siguiente texto con le...,7.4372,success,None,{},881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_refinement_m20_20260321_171527,6fa8b84e-4ddf-474d-a9b5-0fb0b0f90bff,2026-03-21T17:22:45.087748,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,Un problema importante en los negocios es dete...,Reescribe en español el siguiente texto con le...,7.0582,success,None,{},2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


In [17]:
evaluated_df = evaluate_dataframe(
    successful_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
)

print("Shape evaluated_df:", evaluated_df.shape)
display(evaluated_df.head(3))

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

Shape evaluated_df: (320, 51)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,exp_refinement_m20_20260321_171527,ac09fbe2-f126-47cc-8d1f-c4e58ee7ac4c,2026-03-21T17:22:27.934160,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.222222,0.300000,52.468333,34.855000,17.613333,0.736842,0.705882,0.736842,0.912748,None
1,exp_refinement_m20_20260321_171527,b0cd4bff-0834-4169-85f8-33278c9b71c4,2026-03-21T17:22:37.517136,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.428571,0.500000,112.735000,105.172500,7.562500,0.333333,0.142857,0.266667,0.757445,None
2,exp_refinement_m20_20260321_171527,6fa8b84e-4ddf-474d-a9b5-0fb0b0f90bff,2026-03-21T17:22:45.087748,muestra_20_refinamiento,None,refinement,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.714286,0.764706,59.335000,76.229118,-16.894118,0.461538,0.250000,0.307692,0.848482,None


In [23]:
param_cols = evaluated_df["generation_config"].apply(pd.Series)
param_cols = param_cols.rename(columns=lambda c: f"gen_{c}")

evaluated_df = pd.concat([evaluated_df, param_cols], axis=1)

display(
    evaluated_df[
        [
            "model_key",
            "prompt_type",
            "ruleset",
            "gen_temperature",
            "gen_top_p",
            "gen_repetition_penalty",
            "gen_max_new_tokens",
        ]
    ].head(5)
)

,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens
0,llama3,zero-shot,R0,0.2,0.85,1.1,256.0
1,llama3,zero-shot,R0,0.2,0.85,1.1,256.0
2,llama3,zero-shot,R0,0.2,0.85,1.1,256.0
3,llama3,zero-shot,R0,0.2,0.85,1.1,256.0
4,llama3,zero-shot,R0,0.2,0.85,1.1,256.0


In [24]:
evaluated_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_evaluated.csv"

evaluated_df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(evaluated_path)

Archivo guardado en:
/home/harielpadillasanchez/Documentos/TT/TT2/outputs/refinement_muestra_20/exp_refinement_m20_20260321_171527_evaluated.csv


In [25]:
summary_by_model = summarize_metrics(
    evaluated_df,
    group_cols=["model_key"],
)

display(summary_by_model)

summary_by_model_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_summary_by_model.csv"
summary_by_model.to_csv(summary_by_model_path, index=False, encoding="utf-8-sig")

print(summary_by_model_path)

,model_key,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,40.148119,12.229411,83.960127,88.314342,-4.354215,0.787810,0.71250,0.462314,0.0,0.424931,0.552695,0.426835,0.239964,0.362849,70.221505,51.983615,18.237890,0.792289
1,mistral,40.006166,11.989589,82.547272,88.314342,-5.767070,1.272307,0.86875,0.436241,0.0,0.454101,0.402706,0.442309,0.250413,0.376738,60.071203,51.983615,8.087588,0.788227


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/refinement_muestra_20/exp_refinement_m20_20260321_171527_summary_by_model.csv


In [26]:
summary_by_full_config = summarize_metrics(
    evaluated_df,
    group_cols=[
        "model_key",
        "prompt_type",
        "ruleset",
        "gen_temperature",
        "gen_top_p",
        "gen_repetition_penalty",
        "gen_max_new_tokens",
    ],
)

display(
    summary_by_full_config.sort_values(
        by=["sari", "bertscore_f1", "rougeL_f"],
        ascending=False
    ).head(20)
)

summary_by_full_config_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_summary_by_full_config.csv"
summary_by_full_config.to_csv(summary_by_full_config_path, index=False, encoding="utf-8-sig")

print(summary_by_full_config_path)

,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
2,llama3,few-shot,R4,0.2,0.85,1.10,256.0,43.066691,16.676595,83.989712,...,0.0,0.356878,0.500578,0.454335,0.288369,0.396049,70.097098,51.983615,18.113484,0.797879
14,mistral,zero-shot,R4,0.2,0.85,1.10,256.0,41.820725,14.111451,81.347294,...,0.0,0.381495,0.437263,0.475363,0.286722,0.400569,63.757257,51.983615,11.773642,0.801454
3,llama3,few-shot,R4,0.3,0.90,1.15,256.0,41.645703,12.850460,84.425256,...,0.0,0.428751,0.552830,0.438498,0.240393,0.367833,69.177805,51.983615,17.194190,0.794220
1,llama3,few-shot,R0,0.3,0.90,1.15,256.0,41.349748,13.851484,83.203930,...,0.0,0.364905,0.497782,0.448341,0.264478,0.380868,66.653139,51.983615,14.669525,0.803657
13,mistral,zero-shot,R0,0.3,0.90,1.15,256.0,40.832263,12.831806,84.517590,...,0.0,0.407638,0.449929,0.456487,0.258887,0.393376,64.460559,51.983615,12.476944,0.791794
8,mistral,few-shot,R0,0.2,0.85,1.10,256.0,40.674989,13.454886,79.852024,...,0.0,0.457222,0.346205,0.472694,0.280826,0.407538,54.640724,51.983615,2.657109,0.794743
6,llama3,zero-shot,R4,0.2,0.85,1.10,256.0,40.412486,11.292995,84.696719,...,0.0,0.449179,0.560282,0.445301,0.244334,0.384154,72.003265,51.983615,20.019650,0.797336
0,llama3,few-shot,R0,0.2,0.85,1.10,256.0,40.096030,14.038280,82.637409,...,0.0,0.320451,0.494287,0.448439,0.265983,0.382961,68.329113,51.983615,16.345498,0.798674
15,mistral,zero-shot,R4,0.3,0.90,1.15,256.0,39.923263,11.279161,82.943020,...,0.0,0.442634,0.462536,0.424815,0.238251,0.347627,66.340619,51.983615,14.357004,0.789426
11,mistral,few-shot,R4,0.3,0.90,1.15,256.0,39.848115,10.217628,81.116678,...,0.0,0.543035,0.371327,0.410224,0.219793,0.346514,52.538814,51.983615,0.555199,0.773518


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/refinement_muestra_20/exp_refinement_m20_20260321_171527_summary_by_full_config.csv


In [27]:
qual_cols = [
    "sample_id",
    "idFinal",
    "grupo",
    "motivo",
    "model_key",
    "prompt_type",
    "ruleset",
    "gen_temperature",
    "gen_top_p",
    "gen_repetition_penalty",
    "gen_max_new_tokens",
    "source_text",
    "reference_text",
    "generated_text",
    "sari",
    "bertscore_f1",
    "rougeL_f",
    "compression_ratio_eval",
    "exact_copy",
]

qual_cols = [c for c in qual_cols if c in evaluated_df.columns]

qual_review_df = evaluated_df[qual_cols].copy()

display(qual_review_df.head(10))

qual_review_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_qualitative_review.csv"
qual_review_df.to_csv(qual_review_path, index=False, encoding="utf-8-sig")

print(qual_review_path)

,sample_id,idFinal,grupo,motivo,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,source_text,reference_text,generated_text,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,zero-shot,R0,0.2,0.85,1.1,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,67.760943,0.912748,0.736842,0.900000,0
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,llama3,zero-shot,R0,0.2,0.85,1.1,256.0,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Pon $1 al día y también mete el cambio en un s...,36.672526,0.757445,0.266667,0.875000,0
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,llama3,zero-shot,R0,0.2,0.85,1.1,256.0,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,Un problema importante en los negocios es dete...,42.053571,0.848482,0.307692,0.823529,0
3,3679,829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,llama3,zero-shot,R0,0.2,0.85,1.1,256.0,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",El precio del kilo de pan incluye un impuesto ...,47.099192,0.759468,0.322581,0.733333,0
4,3145,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,llama3,zero-shot,R0,0.2,0.85,1.1,256.0,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,La empresa paga $2.20 por cada $100 que vende.,37.904610,0.775162,0.320000,0.909091,0
5,507,562_LibroUAC_Sincopyright.pdf,B_medianos,Porcentajes y rentabilidad.,llama3,zero-shot,R0,0.2,0.85,1.1,256.0,"En general, se dice que la rentabilidad normal...","Generalmente, la rentabilidad normal de una in...",La rentabilidad normal de una inversión en paí...,56.356424,0.884184,0.571429,0.625000,0
6,1756,5100_LibroBAC.pdf,B_medianos,Redacción abstracta/institucional.,llama3,zero-shot,R0,0.2,0.85,1.1,256.0,En virtud de estas y otras reflexiones fue que...,Debido a estas y otras reflexiones se diseñó e...,Este capítulo fue creado como resultado de est...,34.893866,0.757172,0.222222,0.480000,0
7,3093,875_LibroUide_Sincopyright.pdf,B_medianos,Definición contable.,llama3,zero-shot,R0,0.2,0.85,1.1,256.0,Las cuentas de activo reflejan aquello que pos...,Las cuentas de activo evidencian las pertenenc...,La cuenta de activo muestra lo que una empresa...,38.781130,0.824292,0.312500,0.937500,0
8,3192,1202_LibroUide_Sincopyright.pdf,B_medianos,"Relación de pasivo/activo, útil para precisión.",llama3,zero-shot,R0,0.2,0.85,1.1,256.0,"Por cada dólar de pasivo corriente, la compañí...",La empresa tiene noventa y tres centavos de dó...,Para cada dólar de deudas que debe pagar pront...,28.435722,0.755987,0.190476,0.947368,0
9,3525,230_LibroUAC_Sincopyright.pdf,B_medianos,Dato económico con cifra.,llama3,zero-shot,R0,0.2,0.85,1.1,256.0,Posee la renta per cápita más alta de Latinoam...,Tiene la renta per cápita más alta de Latinoam...,La renta per cápita en este país es la más alt...,39.847340,0.795133,0.553191,1.047619,0


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/refinement_muestra_20/exp_refinement_m20_20260321_171527_qualitative_review.csv


In [28]:
best_cases = qual_review_df.sort_values(by="sari", ascending=False).head(15)
display(best_cases)

,sample_id,idFinal,grupo,motivo,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,source_text,reference_text,generated_text,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy
180,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",mistral,zero-shot,R0,0.3,0.90,1.15,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,"La comunidad humana más antigua se llama ""hord...",69.184103,0.893084,0.736842,0.900000,0
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,zero-shot,R0,0.2,0.85,1.10,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,67.760943,0.912748,0.736842,0.900000,0
20,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,zero-shot,R0,0.3,0.90,1.15,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,67.760943,0.912748,0.736842,0.900000,0
140,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,few-shot,R4,0.3,0.90,1.15,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,67.760943,0.912748,0.736842,0.900000,0
120,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,few-shot,R4,0.2,0.85,1.10,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,67.760943,0.912748,0.736842,0.900000,0
100,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,few-shot,R0,0.3,0.90,1.15,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,67.760943,0.912748,0.736842,0.900000,0
260,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",mistral,few-shot,R0,0.3,0.90,1.15,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se ha llamado ...,66.815777,0.877044,0.700000,1.000000,0
240,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",mistral,few-shot,R0,0.2,0.85,1.10,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se ha llamado ...,66.815777,0.877044,0.700000,1.000000,0
200,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",mistral,zero-shot,R4,0.2,0.85,1.10,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se ha llamado ...,66.815777,0.877044,0.700000,1.000000,0
80,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,few-shot,R0,0.2,0.85,1.10,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se conoce como...,66.491101,0.898489,0.700000,1.000000,0


In [29]:
worst_cases = qual_review_df.sort_values(by="sari", ascending=True).head(15)
display(worst_cases)

,sample_id,idFinal,grupo,motivo,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,source_text,reference_text,generated_text,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy
258,2523,282a_LibroNEFE_Sincopyright.pdf,D_complejos,Enumeración larga con información personal/cre...,mistral,few-shot,R0,0.2,0.85,1.10,256.0,"Además de tus datos personales (nombre, direcc...","El informe contiene tus datos personales, tu i...","Además de tus datos personales (nombre, direcc...",22.000914,0.767327,0.407960,0.929688,0
298,2523,282a_LibroNEFE_Sincopyright.pdf,D_complejos,Enumeración larga con información personal/cre...,mistral,few-shot,R4,0.2,0.85,1.10,256.0,"Además de tus datos personales (nombre, direcc...","El informe contiene tus datos personales, tu i...","Además de tus datos personales (nombre, direcc...",23.717256,0.757752,0.373737,0.921875,0
318,2523,282a_LibroNEFE_Sincopyright.pdf,D_complejos,Enumeración larga con información personal/cre...,mistral,few-shot,R4,0.3,0.90,1.15,256.0,"Además de tus datos personales (nombre, direcc...","El informe contiene tus datos personales, tu i...","Además de tus datos personales (nombre, direcc...",24.039562,0.787665,0.391960,0.914062,0
278,2523,282a_LibroNEFE_Sincopyright.pdf,D_complejos,Enumeración larga con información personal/cre...,mistral,few-shot,R0,0.3,0.90,1.15,256.0,"Además de tus datos personales (nombre, direcc...","El informe contiene tus datos personales, tu i...","Además de tus datos personales (nombre, direcc...",25.476272,0.774331,0.390000,0.921875,0
81,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,llama3,few-shot,R0,0.2,0.85,1.10,256.0,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Prueba a guardar $1 al día y también guarden e...,26.278078,0.743202,0.074074,0.687500,0
101,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,llama3,few-shot,R0,0.3,0.90,1.15,256.0,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Prueba a guardar cada día una moneda de $1 y t...,26.713506,0.771627,0.129032,0.937500,0
68,3192,1202_LibroUide_Sincopyright.pdf,B_medianos,"Relación de pasivo/activo, útil para precisión.",llama3,zero-shot,R4,0.3,0.90,1.15,256.0,"Por cada dólar de pasivo corriente, la compañí...",La empresa tiene noventa y tres centavos de dó...,"Para cada dólar que debe ser pagado pronto, la...",26.718575,0.747412,0.195122,0.894737,0
32,329,17510_LibroBAC.pdf,C_estructurales,Definición tipo glosario.,llama3,zero-shot,R0,0.3,0.90,1.15,256.0,Arancel: tarifa oficial que determina los dere...,El arancel es una tarifa oficial y determina l...,Impuesto: una cantidad de dinero que se debe p...,27.103432,0.753521,0.140351,0.846154,0
12,329,17510_LibroBAC.pdf,C_estructurales,Definición tipo glosario.,llama3,zero-shot,R0,0.2,0.85,1.10,256.0,Arancel: tarifa oficial que determina los dere...,El arancel es una tarifa oficial y determina l...,Impuesto: una cantidad de dinero que se debe p...,27.147523,0.750917,0.145455,0.807692,0
103,3679,829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,llama3,few-shot,R0,0.3,0.90,1.15,256.0,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",El costo total por kilogramo de pan es de $190.,27.581284,0.731989,0.200000,0.666667,0


## Resumen de resultados

En esta fase se hizo un refinamiento inicial usando una muestra de 20 ejemplos representativos del dataset.  
La idea fue probar distintas combinaciones de:

- modelos (`llama3` y `mistral`)
- tecnicas (`zero-shot` y `few-shot`)
- rulesets (`R0` y `R4`)
- hiperparametros (2 configuraciones)

En total se ejecutaron **320 corridas** y todas terminaron correctamente, asi que el flujo completo del notebook ya quedo funcional para esta etapa.

### Lo mas importante que se observo

De manera general, **llama3 obtuvo los mejores resultados**, sobre todo cuando se uso con la tecnica **few-shot** y con el ruleset **R4**, que corresponde al conjunto mas completo de reglas.  
Esto sugiere que el modelo responde mejor cuando recibe ejemplos previos y una guia de simplificacion mas rica.

La mejor configuracion encontrada en esta fase fue:

- **modelo:** `llama3`
- **tecnica:** `few-shot`
- **ruleset:** `R4`
- **temperature:** `0.2`
- **top_p:** `0.85`
- **repetition_penalty:** `1.10`
- **max_new_tokens:** `256`

Con esta combinacion se obtuvieron los mejores valores globales en las metricas principales, especialmente en **SARI** y **BERTScore**, por lo que se considera la configuracion candidata para la siguiente fase.

### Interpretacion general

Los resultados muestran que:

- **few-shot ayudo mas que zero-shot** en llama3
- **R4 funciono mejor que R0** en la mayoria de los casos
- `mistral` tambien dio resultados utiles, pero en general quedo por debajo de `llama3`
- la configuracion con temperatura mas baja parecio dar respuestas mas estables y mas alineadas con la simplificacion esperada

Tambien se observo en la revision cualitativa que algunos textos generados por el modelo si simplifican la redaccion, pero no siempre hacen el mismo cambio lexico que aparece en la referencia. O sea, a veces el modelo conserva palabras del original y simplifica mas la estructura que el vocabulario.

### Conclusión de esta fase

Esta etapa sirvio para depurar el notebook, validar que la arquitectura experimental ya funciona de extremo a extremo y seleccionar una configuracion base para pruebas mas grandes.

Con base en estos resultados, la siguiente fase deberia enfocarse en usar como punto de partida:

- `llama3`
- `few-shot`
- hiperparametros ganadores
- y ahora si comparar de forma mas completa los rulesets `R0`, `R1`, `R2`, `R3` y `R4`
